## Importar librerías e instalar paquetes

In [28]:
import torch
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoFeatureExtractor

from torchvision import transforms
from tqdm import tqdm
import sklearn.metrics as metrics
import pandas as pd

## Usar GPU

In [29]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cpu


## Características de cada modelo

In [30]:
model_image_sizes = {
    "swinv2": 256,
    "beit": 224,
    "vit": 224,
    "convnext": 224,
    "dinov2": 224
}

## Cargar dataset de prueba

In [31]:
dataset = load_from_disk("../dataset")
test_dataset = dataset["test"]

In [32]:
print("Columnas:", test_dataset.column_names)
print("Cant. de imgs de test:", len(test_dataset))

Columnas: ['label', 'image', 'video', 'id']
Cant. de imgs de test: 24040


## Función de evaluación de los modelos

In [33]:
def evaluate_model(model_path, test_dataset, image_size):
    print(f"\nEvaluando modelo: {model_path}")
    
    # Cargar modelo y extractor
    model = AutoModelForImageClassification.from_pretrained(model_path).to(DEVICE)
    extractor = AutoFeatureExtractor.from_pretrained(model_path)
    
    # Transformar datos
    _test_transform = get_test_transform(image_size)
    
    def test_transform(ex):
        ex["pixel_values"] = [_test_transform(img) for img in ex["image"]]
        return ex
    
    test_dataset.set_transform(test_transform)
    
    # Crear dataloader
    def collate_fn(examples):
        pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
        labels = torch.tensor([ex["label"] for ex in examples])
        return {"pixel_values": pixel_values, "labels": labels}
    
    test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)
    
    # Evaluar
    preds, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluando"):
            pixel_values = batch["pixel_values"].to(DEVICE)
            batch_labels = batch["labels"].to(DEVICE)
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch_labels.cpu().numpy())
    
    # Calcular métricas
    acc = metrics.accuracy_score(labels, preds)
    f1 = metrics.f1_score(labels, preds, average="weighted")
    
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    return {"model": model_path, "accuracy": acc, "f1": f1}

In [34]:
results = []

In [35]:
for model_name in model_image_sizes.keys():
    size = next((v for k, v in model_image_sizes.items() if k in short_name.lower()), 224)
    result = evaluate_model(f"./models/{short_name}", test_dataset, size)
    results.append(result)


Evaluando modelo: ./models/swinv2-base-patch4-window16-256


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './models/swinv2-base-patch4-window16-256'. Use `repo_type` argument if needed.

## Resultados de evaluación de los modelos

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results